## ONLY RUN THIS AFTER PROCESS_PBP_DATA.IPYNB (requires the ao ids from players.csv)

### The atp id matches the end of the australian open id, so players that have it can have their ranking history automaticaly ported over 

### The missing players need their id filled manually, since their is no reliable API for that! (this can be used to help: https://www.atptour.com/en/-/www/site-search/SOCK/)

#### Use this 'API' to find the PlayerId and add it to the players.csv in player_id_ao -> 'ATP' + PlayerId : https://www.atptour.com/en/-/www/site-search/{player_name}/

## Script for scraping rankings from the ATP website

In [14]:
import pandas as pd
import numpy as np
import json
import os
import re
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from collections import Counter


pd.set_option('display.max_rows', 500)

# --> Import functions from "process" script
import sys
sys.path.append('../src')

from search_utils import get_player_ranking_history, get_all_players_info_df

In [ ]:
ranking_dir = 'F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/'

In [16]:

def scrape_atp_rank_history(player_id):
    options = Options()
    options.headless = True
    driver = webdriver.Chrome(service=Service('F:/U3/chromedriver-win64/chromedriver-win64/chromedriver.exe'), options=options)
    api_url = f'https://www.atptour.com/en/-/www/rank/history/{player_id}?v=1'

    try:
        driver.get(api_url)

        # Get the page source which contains the JSON data
        page_source = driver.page_source

        # Use regular expressions to find the JSON data within the page source
        json_data_match = re.search(r'<pre.*?>(.*?)</pre>', page_source, re.DOTALL)
        if json_data_match:
            # Extract the JSON string
            json_data = json_data_match.group(1)

            # Convert the JSON string to a dictionary
            rank_history = json.loads(json_data)

            print(f'Rank history for player {player_id}: {rank_history}')
            return rank_history
        else:
            print(f'JSON data not found for player {player_id}')
    except Exception as e:
        print(f'Failed to retrieve data for player {player_id}: {e}')
    finally:
        driver.quit()

def save_atp_players_rank_history(players_ids_atp, league):
    dir = ranking_dir
    count = 0
    for player_id in players_ids_atp:
        count+=1
        print("count:", count) 
        print('player_id: ', player_id)
        file_name = f"{league}_{player_id}_player_ranking_history.json"
        file_name = dir + file_name
        # Check if the file already exists
        if os.path.exists(file_name):
            print(f"Skipping {file_name}. File already exists.")
            continue

        rank_history = scrape_atp_rank_history(player_id)
        print(rank_history['FirstRankYear'])

        with open(file_name, 'w') as json_file:
            json.dump(rank_history, json_file, indent=4)

In [17]:
league = 'atp'

players_info_df = get_all_players_info_df('', league)

print(players_info_df)
players_with_miss_id = players_info_df['player_name'][players_info_df['player_id_atp'].isna()]

atp_ids = np.array(players_info_df['player_id_atp'])

players_with_miss_id = np.array(players_with_miss_id)

             player_name player_id_ao league  player_id_rg country  \
1               A.BALAZS      ATPBD80    atp         15374     HUN   
3               A.BEDENE      ATPBH09    atp         19666     SLO   
7                 A.BOLT      ATPBI81    atp             0     NaN   
9               A.BUBLIK      ATPBK92    atp         29098     KAZ   
10              A.CAZAUX      ATPC0H0    atp         43952     FRA   
12    A.DAVIDOVICHFOKINA      ATPDH50    atp         39036     ESP   
13            A.DEMINAUR      ATPDH58    atp         36276     AUS   
14                A.FILS      ATPF0F1    atp         47762     FRA   
16           A.GIANNESSI          NaN    atp         18839     ITA   
17              A.HARRIS      ATPHB29    atp             0     NaN   
19               A.HOANG          NaN    atp         27071     FRA   
22            A.KARATSEV      ATPKC56    atp         24955     ---   
27           A.KOVACEVIC      ATPK0AZ    atp         41293     USA   
30           A.KUZNE

In [19]:
names = players_info_df['player_name']
names

In [20]:
len(names)

266

In [21]:
players_info_df = get_all_players_info_df('', league)
players_with_miss_id = players_info_df['player_name'][players_info_df['player_id_atp'].isna()]

players_with_miss_id = np.array(players_with_miss_id)
if not any(player is None for player in players_with_miss_id):
    print("No missing ATP IDs found.")
else:
    print("Missing ATP IDs found:", players_with_miss_id)

    print("Use this 'API' to find the PlayerId and add it to the players.csv in player_id_atp: https://www.atptour.com/en/-/www/site-search/SOCK/")
    raise ValueError("One or more players have a missing ID -> Players with missing ATP id: " + str(players_with_miss_id))

print("Players with missing atp id: ",  players_with_miss_id)

#See process_pbp_data


No missing ATP IDs found.
Players with missing atp id:  []


In [22]:
len(atp_ids)

266

In [23]:
id_counts = Counter(atp_ids)

repeated_ids = [id for id, count in id_counts.items() if count > 1]

if repeated_ids:
    print("The following strings are repeated:")
    for id in repeated_ids:
        print(id)
else:
    print("There are no repeated strings in the array.")


There are no repeated strings in the array.


In [24]:
atp_ids

array(['BD80', 'BH09', 'BI81', 'BK92', 'C0H0', 'DH50', 'DH58', 'F0F1',
       'G983', 'HB29', 'HA71', 'KC56', 'K0AZ', 'KB54', 'ME82', 'MF35',
       'M0QI', 'MV14', 'MP20', 'MC10', 'P09Z', 'R772', 'RC91', 'RE44',
       'SA93', 'S0H2', 'TE30', 'V832', 'W09E', 'Z355', 'BM95', 'CG80',
       'F811', 'GH92', 'H09P', 'N0AE', 'PD31', 'S0S1', 'TA46', 'V812',
       'Z419', 'A0E2', 'E865', 'GD64', 'LB66', 'MW02', 'N771', 'O483',
       'RH16', 'SK94', 'TE16', 'T0AP', 'U182', 'AE14', 'D923', 'E687',
       'GE33', 'GB88', 'I165', 'KE73', 'KB09', 'L987', 'MM58', 'DB59',
       'P0HW', 'SM37', 'SU55', 'S0GD', 'S0IA', 'TB69', 'BT68', 'CG94',
       'D864', 'E873', 'GC89', 'GA36', 'G858', 'RH24', 'Y218', 'AA27',
       'AG37', 'BF23', 'C0AU', 'C0E9', 'CE77', 'D874', 'D0CG', 'SN54',
       'F510', 'KB05', 'L397', 'M0CI', 'MW75', 'TD51', 'V306', 'BK24',
       'BU54', 'D875', 'ML57', 'MC65', 'PC11', 'SD32', 'Z0A1', 'O660',
       'DA31', 'G09O', 'GF95', 'HB71', 'L949', 'R0DG', 'I305', 'K336',
      

In [29]:
save_atp_players_rank_history(atp_ids, 'atp')

count: 1
player_id:  BD80
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_BD80_player_ranking_history.json. File already exists.
count: 2
player_id:  BH09
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_BH09_player_ranking_history.json. File already exists.
count: 3
player_id:  BI81
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_BI81_player_ranking_history.json. File already exists.
count: 4
player_id:  BK92
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_BK92_player_ranking_history.json. File already exists.
count: 5
player_id:  C0H0
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_C0H0_player_ranking_history.json. File already exists.
count: 6
player_id:  DH50
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_DH50_player_ranking_history.json. File already exists.
count: 7
player_id:  DH58
Skipping F:/U3/TCC/TennisCCH/all_json/ranking_history/atp/atp_DH58_player_ranking_history.json. File already exists.

In [27]:
player_hist = get_player_ranking_history('D643', 'atp')

In [28]:
player_hist.head()

,rank_date,singles_roll_rank,singles_roll_tie,singles_roll_points,singles_race_rank,singles_race_tie,singles_race_points,doubles_roll_rank,doubles_roll_tie,doubles_roll_points,first_rank_year,last_rank_year
0,27-05-2024,1,False,9960,12,False,1460,0,False,0,2003,2024
1,20-05-2024,1,False,9860,14,False,1360,0,False,0,2003,2024
2,06-05-2024,1,False,9990,12,False,1310,0,False,0,2003,2024
3,22-04-2024,1,False,9990,9,False,1310,0,False,0,2003,2024
4,15-04-2024,1,False,10035,9,False,1310,0,False,0,2003,2024
